In [ ]:
import numpy as np

K = 5
M = 3

alpha = 0.001  # Learning rate
batch_size = 2  # Mini-batch size

N_0 = 1e-9
P_t = 10

h_k = np.random.rand(M, K)
P = np.random.rand(K, M)
c = np.random.rand(K)
u = np.random.rand(K)
gamma_k = np.random.rand(K)

Rth_s = np.random.rand(K)
w_secrecy = 10

num_iterations = 1000

def secrecy_rate(P, h_k, gamma_k, N_0):
    K, M = P.shape
    R_s = np.zeros(K)
    for k in range(K):
        h_k_user = h_k[:, k]
        P_k = P[k, :]
        interference = np.sum(np.abs(np.dot(h_k.T, P_k)) ** 2) - np.abs(np.dot(h_k_user.T, P_k))**2
        R_s[k] = np.log2(1 + np.abs(np.dot(h_k_user, P_k))**2 / (N_0 + interference))
    return R_s

def transmit_power_constraint(P, P_t):
    power = np.trace(np.dot(P, P.T))
    return power <= P_t

def non_negativity_check(c):
    return np.all(c >= 0)

penalty_factor = 100

# SGD Loop
for t in range(1, num_iterations+1):
    grad_P = np.zeros_like(P)
    grad_c = np.zeros_like(c)

    # Stochastic Gradient Descent: Update with a mini-batch of random indices
    indices = np.random.choice(K, batch_size, replace=False)
    for k in indices:
        grad_R_s_k = (1 / np.log(2)) * (2 * np.real(np.dot(h_k[:, k].T, P[k]))) / (
            (1 + gamma_k[k]) * (np.sum(np.abs(np.dot(h_k.T, P[k])) ** 2) + N_0))
        grad_P[k] = u[k] * grad_R_s_k + w_secrecy * grad_R_s_k * 2
        grad_c[k] = u[k] + w_secrecy * grad_R_s_k

    # Update parameters with mini-batch gradients
    P = P - alpha * grad_P
    c = c - alpha * grad_c

    R_s = secrecy_rate(P, h_k, gamma_k, N_0)
    secrecy_penalty = np.sum(np.maximum(0, Rth_s - R_s))
    total_loss = secrecy_penalty + penalty_factor * secrecy_penalty

    if t % 100 == 0:
        print(f"Secrecy Rate Penalty: {secrecy_penalty}")

    secrecy_constraint_satisfied = np.all(R_s >= Rth_s)
    power_constraint_satisfied = transmit_power_constraint(P, P_t)
    non_negativity_satisfied = non_negativity_check(c)

    if t % 100 == 0:
        print(f"Transmit Power Constraint Satisfied: {power_constraint_satisfied}")
        print(f"Non-Negativity of Rate Allocation Satisfied: {non_negativity_satisfied}")

print("\nFinal optimized precoder matrix (P):")
print(P)

print("\nFinal optimized rate allocation vector (c):")
print(c)


Secrecy Rate Penalty: 2.4134431327260795
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: False
Secrecy Rate Penalty: 2.8279845300756437
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: False
Secrecy Rate Penalty: 2.8506293611886586
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: False
Secrecy Rate Penalty: 2.850632023900152
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: False
Secrecy Rate Penalty: 2.850632025261003
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: False
Secrecy Rate Penalty: 2.8506320252612047
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: False
Secrecy Rate Penalty: 2.850632025261205
Transmit Power Constraint Satisfied: True
Non-Negativity of Rate Allocation Satisfied: False
Secrecy Rate Penalty: 2.850632025261205
Transmit Power Constraint